# NN-01 Forward Pass

- Module: 06 Neural Networks
- Topic: explicit affine maps and activations in a multilayer perceptron
- Estimated runtime: 5 minutes
- Prerequisites: matrix multiplication, elementwise nonlinearities, basic NumPy
- Expected outputs: layer-by-layer activations, shape checks, and agreement with the reusable scratch implementation


## Learning Goals

By the end of this notebook, you should be able to:
- compute a forward pass one layer at a time with explicit matrix operations;
- track the shapes of inputs, weights, biases, pre-activations, and activations; and
- connect the manual computation to the reusable `ScratchMLP.forward` implementation.


In [ ]:
from __future__ import annotations

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from notebook path.')


REPO_ROOT = find_repo_root(Path.cwd())
MODULE_SRC = REPO_ROOT / 'modules/06-neural-networks/src'
if str(MODULE_SRC) not in sys.path:
    sys.path.insert(0, str(MODULE_SRC))

from nn_from_scratch import ScratchMLP, train_test_split_indices

SEED = 7
rng = np.random.default_rng(SEED)
SEED


## Outline

1. Build a tiny two-layer network with hand-chosen parameters.
2. Compute `Z^{[1]}`, `A^{[1]}`, `Z^{[2]}`, and `A^{[2]}` explicitly.
3. Compare the manual result against the library forward pass.
4. Inspect the cached tensors that backpropagation will later reuse.


## Step 1 - Fix a small batch and explicit parameters

The batch uses two examples, each with two input features. We keep the network small enough that every matrix is readable on screen.


In [ ]:
X = np.array(
    [
        [0.20, -0.40],
        [1.10, 0.30],
    ],
    dtype=float,
)

W1 = np.array(
    [
        [0.80, -0.50],
        [0.30, 0.90],
        [-0.40, 0.70],
    ],
    dtype=float,
)
b1 = np.array([[0.10, -0.20, 0.05]], dtype=float)
W2 = np.array([[1.20, -0.70, 0.40]], dtype=float)
b2 = np.array([[0.15]], dtype=float)

shape_summary = {
    'X': X.shape,
    'W1': W1.shape,
    'b1': b1.shape,
    'W2': W2.shape,
    'b2': b2.shape,
}
shape_summary


## Step 2 - Compute the hidden layer explicitly

For a batch matrix $X \in \mathbb{R}^{n \times d_0}$ and a weight matrix $W^{[1]} \in \mathbb{R}^{d_1 \times d_0}$,

$$
Z^{[1]} = X (W^{[1]})^\top + b^{[1]}, \qquad A^{[1]} = \tanh(Z^{[1]}).
$$

The bias broadcasts across the batch dimension.


In [ ]:
# X: (n, d0), W1: (d1, d0), b1: (1, d1), Z1: (n, d1)
Z1 = X @ W1.T + b1
A1 = np.tanh(Z1)

{
    'Z1': np.round(Z1, 4),
    'A1': np.round(A1, 4),
    'Z1_shape': Z1.shape,
    'A1_shape': A1.shape,
}


## Step 3 - Compute the output layer explicitly

The second affine map consumes the hidden activations and produces a single logit per example. We then apply a sigmoid to interpret the result as a probability.


In [ ]:
# A1: (n, d1), W2: (d2, d1), b2: (1, d2), Z2: (n, d2)
Z2 = A1 @ W2.T + b2
A2 = 1.0 / (1.0 + np.exp(-Z2))

{
    'Z2': np.round(Z2, 4),
    'A2': np.round(A2, 4),
    'Z2_shape': Z2.shape,
    'A2_shape': A2.shape,
}


## Step 4 - Compare against `ScratchMLP.forward`

The library function should produce the same activations and also cache the intermediate arrays needed for backpropagation.


In [ ]:
model = ScratchMLP(layer_dims=[2, 3, 1], activations=['tanh', 'sigmoid'], loss='binary_cross_entropy', seed=SEED)
model.parameters['W1'] = W1.copy()
model.parameters['b1'] = b1.copy()
model.parameters['W2'] = W2.copy()
model.parameters['b2'] = b2.copy()

predictions, caches = model.forward(X)

comparison = {
    'manual_matches_model': bool(np.allclose(A2, predictions)),
    'hidden_matches_cache': bool(np.allclose(A1, caches[0]['A'])),
    'output_matches_cache': bool(np.allclose(A2, caches[1]['A'])),
}
comparison


## Interpretation

The forward pass is only repeated affine transformation plus nonlinearity. The key bookkeeping burden is shape discipline, not conceptual mystery.


In [ ]:
cache_report = []
for layer_number, cache in enumerate(caches, start=1):
    cache_report.append(
        {
            'layer': layer_number,
            'A_prev': cache['A_prev'].shape,
            'Z': cache['Z'].shape,
            'A': cache['A'].shape,
            'W': cache['W'].shape,
            'b': cache['b'].shape,
            'activation': str(cache['activation'][0]),
        }
    )
cache_report


## Exercise

Change the hidden activation from `tanh` to `relu` and recompute the forward pass. Which hidden units become exactly zero, and how does that change the output probability?


In [ ]:
relu_model = ScratchMLP(layer_dims=[2, 3, 1], activations=['relu', 'sigmoid'], loss='binary_cross_entropy', seed=SEED)
relu_model.parameters['W1'] = W1.copy()
relu_model.parameters['b1'] = b1.copy()
relu_model.parameters['W2'] = W2.copy()
relu_model.parameters['b2'] = b2.copy()

relu_predictions, relu_caches = relu_model.forward(X)
{
    'relu_hidden': np.round(relu_caches[0]['A'], 4),
    'relu_output': np.round(relu_predictions, 4),
}


## Summary

You manually computed a full forward pass and verified that the reusable library performs the same matrix operations. The cached values now provide the starting point for backpropagation.
